# MolmoAct2 - LIBERO real-time interactive simulation

Standalone demo: runs **our fine-tuned MolmoAct2 LIBERO policy** in the LIBERO simulator in **real time** and renders it **inline, right here in the notebook** - no training or eval needed, it loads the fine-tuned checkpoint directly.

The simulator steps at wall-clock rate while the planner runs *ahead* of execution and stitches each freshly planned action chunk onto the one still running (RTC "plan-ahead-while-executing"): new motion is ramp-blended over the overlap and the gripper switches with hysteresis, so the arm moves continuously instead of pausing to think (`RT_STITCH=blend`; set `RT_STITCH=hold` to see the stop-and-decide baseline).

There is **no extra port to forward**: the sim's small web UI is served inside your session and proxied through your JupyterHub route via `jupyter-server-proxy`, then embedded below with an `IFrame`. Type an instruction in the panel, press **Send**, and watch the policy act; the status line shows live inference latency, buffer depth, hold %, and accel/jerk smoothness.

**Default weight:** our fine-tuned checkpoint at `~/checkpoints/reference/pretrained_model` (`REFERENCE_POLICY`, staged onto pod storage; not baked into the image). Override with `POLICY_PATH=/path` or a Hub repo id.

**Knobs (env):** `RT_STITCH=blend|hold`, `RT_HZ=20`, `SUITE=libero_object`, `RT_PORT=8080`.

In [ ]:
import os
import subprocess
import json
import time
import urllib.request

# Quiet ROCm/torch spam in any child process we spawn.
os.environ.setdefault("TORCH_BLAS_PREFER_HIPBLASLT", "0")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

OUT_DIR = os.environ.get("OUT_DIR", "/outputs")
os.makedirs(OUT_DIR, exist_ok=True)


# Subprocesses (the sim server) inherit a clean env: drop the notebook's inline matplotlib
# backend (crashes headless children) and force plain hipBLAS + unbuffered output.
def child_env(**extra):
    e = dict(os.environ)
    e.pop("MPLBACKEND", None)
    e.update({"TORCH_BLAS_PREFER_HIPBLASLT": "0", "PYTHONWARNINGS": "ignore", "PYTHONUNBUFFERED": "1"})
    e.update({k: str(v) for k, v in extra.items()})
    return e


# DEFAULT WEIGHT: our fine-tuned LIBERO checkpoint, loaded directly. POLICY_PATH overrides
# (local dir or a Hugging Face repo id).
REFERENCE = os.environ.get("REFERENCE_POLICY", os.path.expanduser("~/checkpoints/reference/pretrained_model"))
_explicit = os.environ.get("POLICY_PATH", "").strip()


def _is_ckpt(p):
    return bool(p) and os.path.isdir(p) and os.path.exists(os.path.join(p, "config.json"))


if _explicit:
    POLICY_PATH = _explicit
    print("interactive sim will load POLICY_PATH (from env):", POLICY_PATH)
elif _is_ckpt(REFERENCE):
    POLICY_PATH = REFERENCE
    print("interactive sim will load our fine-tuned checkpoint (default):", POLICY_PATH)
else:
    raise FileNotFoundError(
        f"Fine-tuned checkpoint not found at {REFERENCE}.\n"
        "Stage our fine-tuned checkpoint there (organizers provide it; it is not baked into the "
        "image), or set POLICY_PATH=/path (or a Hub repo id)."
    )

In [ ]:
from IPython.display import IFrame, display

# Internal port inside your pod; reached only through the authenticated JupyterHub proxy
# (never exposed directly). The server binds 0.0.0.0:RT_PORT; jupyter-server-proxy forwards
# {JUPYTERHUB_SERVICE_PREFIX}/proxy/RT_PORT/ -> 127.0.0.1:RT_PORT.
RT_PORT = os.environ.get("RT_PORT", "8080")
SIM_SERVER = "/ryzers/notebooks/scripts/interactive_server_rt_ft.py"

# Stop a server we started from a previous run of this cell.
try:
    if globals().get("_sim") and _sim.poll() is None:
        _sim.terminate()
        _sim.wait(timeout=5)
except Exception:
    pass

# Serve the fine-tuned policy (POLICY_PATH). Stream server logs to a file so the notebook
# kernel never blocks on a full stdout pipe while the sim runs.
sim_env = child_env(
    POLICY_PATH=POLICY_PATH,
    PORT=RT_PORT,
    SUITE=os.environ.get("SUITE", "libero_object"),
    RT_STITCH=os.environ.get("RT_STITCH", "blend"),  # blend = RTC real-time chunking; hold = stop-and-decide
    RT_HZ=os.environ.get("RT_HZ", "20"),
)
_log_path = os.path.join(OUT_DIR, "interactive_rt.log")
_log = open(_log_path, "w")
_sim = subprocess.Popen(
    ["/opt/train-venv/bin/python", SIM_SERVER],
    env=sim_env, stdout=_log, stderr=subprocess.STDOUT,
)
print(f"real-time sim server started (pid {_sim.pid}); loading policy: {POLICY_PATH}")
print("(first run JIT-compiles kernels; this can take a few minutes)")


def _sim_status():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{RT_PORT}/status", timeout=3) as r:
            return json.load(r)
    except Exception:
        return None


ready, deadline = False, time.time() + 900
while time.time() < deadline:
    if _sim.poll() is not None:
        print("\nserver exited early; tail of log:")
        print("".join(open(_log_path).readlines()[-20:]))
        break
    s = _sim_status()
    if s:
        print("  " + str(s.get("status", ""))[:90], end="\r")
        if s.get("mode") in ("idle", "running"):
            ready = True
            break
        if s.get("mode") == "error":
            print("\nserver error:", s.get("status"))
            break
    time.sleep(3)

prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "").rstrip("/")
sim_url = f"{prefix}/proxy/{RT_PORT}/" if prefix else f"http://localhost:{RT_PORT}/"
if ready:
    print(f"\nready - interactive sim embedded below (also open in a tab: {sim_url})")
    display(IFrame(sim_url, width="100%", height=840))
else:
    print(f"\nserver not ready yet; wait a moment and re-run this cell. URL: {sim_url}")